# 🧠 우울감 → 신체활동 → BMI 매개효과 분석
**데이터**: CDC NHANES 2013–2014 (Kaggle)  
**분석**: Baron & Kenny 4단계 매개효과 + Bootstrap 간접효과 검증

---
### ⚙️ 실행 방법
1. **STEP 0** 셀 실행 → `kaggle.json` 업로드
2. **상단 메뉴 → 런타임 → 모두 실행**
3. 완료까지 약 **3~5분** 소요

## STEP 0. Kaggle 인증

In [ ]:
from google.colab import files
files.upload()  # kaggle.json 선택
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle 인증 완료')

## STEP 1. 환경 설정 + 데이터 다운로드

In [ ]:
# ── 라이브러리 ──────────────────────────────────────────
!pip install -q kaggle pingouin
import subprocess
subprocess.run(['apt-get','-qq','install','fonts-nanum'], check=False)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import pingouin as pg
import warnings, os, glob
warnings.filterwarnings('ignore')

# 한글 폰트
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams.update({
    'axes.unicode_minus': False,
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--'
})

# 컬러 팔레트
C_PURPLE = '#6C5CE7'
C_TEAL   = '#00B894'
C_CORAL  = '#E17055'
C_DARK   = '#2D3436'
C_LIGHT  = '#DFE6E9'

# ── 데이터 다운로드 ─────────────────────────────────────
!kaggle datasets download -d cdc/national-health-and-nutrition-examination-survey --unzip -q
csvs = sorted(glob.glob('*.csv'))
print('다운로드된 파일:', csvs)
print('✅ 환경 준비 완료')

## STEP 2. 핵심 변수 추출 & 병합

In [ ]:
# ── 파일 자동 탐색 ──────────────────────────────────────
def find(keywords):
    for kw in keywords:
        for f in csvs:
            if kw.lower() in f.lower(): return f
    return None

demo_f = find(['demo'])
dpq_f  = find(['dpq','depression','mental'])
paq_f  = find(['paq','physical','activity'])
bmx_f  = find(['bmx','body','exam'])

for name, path in [('인구통계',demo_f),('우울(PHQ-9)',dpq_f),
                    ('신체활동',paq_f),('신체계측/BMI',bmx_f)]:
    print(f'{name}: {path}')

In [ ]:
# ── 각 파일 로드 ────────────────────────────────────────
dfs = {}
for name, path in [('demo',demo_f),('dpq',dpq_f),
                    ('paq',paq_f),('bmx',bmx_f)]:
    if path:
        dfs[name] = pd.read_csv(path)
        print(f'{name}: {dfs[name].shape}  컬럼 예시: {list(dfs[name].columns[:6])}')

# SEQN 기준 병합
from functools import reduce
frames = [df for df in dfs.values()]
id_col = 'SEQN' if 'SEQN' in frames[0].columns else frames[0].columns[0]
raw = reduce(lambda l,r: pd.merge(l,r,on=id_col,how='inner'), frames)
print(f'\n병합 결과: {raw.shape[0]:,}명 × {raw.shape[1]}컬럼')

In [ ]:
# ── 핵심 변수 탐색 ──────────────────────────────────────
cols = raw.columns.str.upper()

def pick(keywords):
    for kw in keywords:
        m = [c for c in cols if kw in c]
        if m: return m[0]
    return None

# PHQ-9 문항 (DPQ010~DPQ090)
phq_items = [c for c in cols if c.startswith('DPQ0') and c[4:].isdigit()]
print('PHQ-9 문항 컬럼:', phq_items)

VAR = {
    'BMI'          : pick(['BMXBMI']),
    'pa_minutes'   : pick(['PAD615','PAQ605']),      # 유산소 운동 분
    'pa_strength'  : pick(['PAD660','PAQ650']),      # 근력 운동 일수
    'age'          : pick(['RIDAGEYR']),
    'sex'          : pick(['RIAGENDR']),
    'income_ratio' : pick(['INDFMPIR']),
    'smoke'        : pick(['SMQ020']),
}

print('\n변수 매핑:')
for k,v in VAR.items():
    print(f'  {k:15s} → {v or "없음"}')
print(f'  PHQ-9 문항     → {len(phq_items)}개')

## STEP 3. 전처리

In [ ]:
df = raw.copy()

# ── PHQ-9 총점 계산 ─────────────────────────────────────
if phq_items:
    # 9점 이상(거부/모름) → NaN 처리
    for c in phq_items:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        df[c] = df[c].where(df[c] <= 3, np.nan)
    df['PHQ9'] = df[phq_items].sum(axis=1, min_count=7)  # 7문항 이상 응답
    print(f'PHQ-9 범위: {df["PHQ9"].min():.0f} ~ {df["PHQ9"].max():.0f}')

# ── 신체활동 통합 지표 ──────────────────────────────────
pa_col = VAR.get('pa_minutes')
ps_col = VAR.get('pa_strength')

if pa_col and pa_col in df.columns:
    df['PA'] = pd.to_numeric(df[pa_col], errors='coerce')
    df['PA'] = df['PA'].where(df['PA'] < 9000, np.nan)  # 9999=모름 제거
elif ps_col and ps_col in df.columns:
    df['PA'] = pd.to_numeric(df[ps_col], errors='coerce')
    df['PA'] = df['PA'].where(df['PA'] < 70, np.nan)

# ── BMI ─────────────────────────────────────────────────
bmi_col = VAR.get('BMI')
if bmi_col:
    df['BMI'] = pd.to_numeric(df[bmi_col], errors='coerce')
    df['BMI'] = df['BMI'].where((df['BMI'] > 10) & (df['BMI'] < 80), np.nan)

# ── 통제변수 ─────────────────────────────────────────────
age_col = VAR.get('age')
sex_col = VAR.get('sex')
inc_col = VAR.get('income_ratio')
smk_col = VAR.get('smoke')

for orig, new in [(age_col,'AGE'),(sex_col,'SEX'),
                   (inc_col,'INCOME'),(smk_col,'SMOKE')]:
    if orig and orig in df.columns:
        df[new] = pd.to_numeric(df[orig], errors='coerce')

# ── 대상자 필터 ─────────────────────────────────────────
if 'AGE' in df.columns:
    df = df[df['AGE'] >= 20]

# ── 분석 데이터프레임 ────────────────────────────────────
core = ['PHQ9','PA','BMI']
covariates = [c for c in ['AGE','SEX','INCOME','SMOKE'] if c in df.columns]
analysis_cols = core + covariates
analysis_cols = [c for c in analysis_cols if c in df.columns]

df_an = df[analysis_cols].dropna().copy()

# ── IQR Winsorize (신체활동, BMI) ────────────────────────
def winsorize(s, q=0.01):
    return s.clip(s.quantile(q), s.quantile(1-q))

df_an['PA']  = winsorize(df_an['PA'])
df_an['BMI'] = winsorize(df_an['BMI'])

# ── 우울 심각도 그룹 ─────────────────────────────────────
def phq_group(s):
    if s <= 4:    return '없음(0-4)'
    elif s <= 9:  return '경미(5-9)'
    elif s <= 14: return '중등도(10-14)'
    else:         return '심각(15+)'

df_an['DEP_GROUP'] = df_an['PHQ9'].apply(phq_group)

print(f'\n✅ 최종 분석 대상: {len(df_an):,}명')
print(df_an[['PHQ9','PA','BMI']].describe().round(2))
print('\n우울 심각도 분포:')
print(df_an['DEP_GROUP'].value_counts())

## STEP 4. 매개효과 분석 — Baron & Kenny 4단계

In [ ]:
# 통제변수 공식 문자열
ctrl = ' + '.join(covariates) if covariates else ''
ctrl_str = f' + {ctrl}' if ctrl else ''

print('=' * 60)
print('📊 Baron & Kenny 4단계 매개효과 검증')
print('=' * 60)
print('독립변수(X): PHQ9 (우울감)')
print('매개변수(M): PA  (신체활동량)')
print('종속변수(Y): BMI (체질량지수)')
print()

results = {}

# ── 1단계: X → Y (총효과) ───────────────────────────────
m1 = smf.ols(f'BMI ~ PHQ9{ctrl_str}', data=df_an).fit()
results['c'] = m1.params['PHQ9']
results['c_p'] = m1.pvalues['PHQ9']
print(f'[1단계] X → Y  β = {m1.params["PHQ9"]:.4f}  p = {m1.pvalues["PHQ9"]:.4f}')
sig = '✔ 유의' if m1.pvalues['PHQ9'] < 0.05 else '✗ 비유의'
print(f'        {sig} → 매개효과 분석 진행 가능')

# ── 2단계: X → M ────────────────────────────────────────
m2 = smf.ols(f'PA ~ PHQ9{ctrl_str}', data=df_an).fit()
results['a'] = m2.params['PHQ9']
results['a_p'] = m2.pvalues['PHQ9']
print(f'\n[2단계] X → M  β = {m2.params["PHQ9"]:.4f}  p = {m2.pvalues["PHQ9"]:.4f}')
print(f'        {"✔ 유의" if m2.pvalues["PHQ9"] < 0.05 else "✗ 비유의"}')

# ── 3단계: X + M → Y ────────────────────────────────────
m3 = smf.ols(f'BMI ~ PHQ9 + PA{ctrl_str}', data=df_an).fit()
results["c'"] = m3.params['PHQ9']
results["c'_p"] = m3.pvalues['PHQ9']
results['b'] = m3.params['PA']
results['b_p'] = m3.pvalues['PA']
print(f"\n[3단계] X+M → Y")
print(f"  PHQ9(X) β = {m3.params['PHQ9']:.4f}  p = {m3.pvalues['PHQ9']:.4f}  {'✔' if m3.pvalues['PHQ9']<0.05 else '✗'}")
print(f"  PA  (M) β = {m3.params['PA']:.4f}  p = {m3.pvalues['PA']:.4f}  {'✔' if m3.pvalues['PA']<0.05 else '✗'}")

# ── 4단계: 간접효과 = a × b ─────────────────────────────
indirect = results['a'] * results['b']
print(f'\n[4단계] 간접효과 (a × b) = {results["a"]:.4f} × {results["b"]:.4f} = {indirect:.4f}')

# 매개 유형 판단
direct_p = results["c'_p"]
if direct_p < 0.05:
    mediation_type = '부분 매개 (Partial Mediation)'
else:
    mediation_type = '완전 매개 (Full Mediation)'
print(f'\n매개 유형: {mediation_type}')
print('=' * 60)

In [ ]:
# ── Bootstrap 간접효과 검증 (pingouin) ──────────────────
print('\n🔁 Bootstrap 간접효과 검증 (n=1000)')
print('  (신뢰구간에 0이 포함되지 않으면 매개효과 유의)')

med_result = pg.mediation_analysis(
    data=df_an,
    x='PHQ9', m='PA', y='BMI',
    covar=covariates if covariates else None,
    n_boot=1000, seed=42
)
print(med_result.to_string())

# 95% CI 추출
indirect_row = med_result[med_result['path'] == 'Indirect']
if len(indirect_row) > 0:
    ci_lo = indirect_row['CI[2.5%]'].values[0]
    ci_hi = indirect_row['CI[97.5%]'].values[0]
    sig_boot = '✅ 유의 (CI에 0 미포함)' if (ci_lo > 0 or ci_hi < 0) else '❌ 비유의 (CI에 0 포함)'
    print(f'\n간접효과 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]  → {sig_boot}')

## STEP 5. 시각화

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 1: 매개효과 경로 다이어그램
# ════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 5)
ax.axis('off')

box_kw = dict(boxstyle='round,pad=0.5', linewidth=1.5)

# 노드
ax.text(1.2, 2.5, '우울감\n(PHQ-9)', ha='center', va='center', fontsize=13,
        fontweight='bold', bbox=dict(**box_kw, facecolor='#E8DAFF', edgecolor=C_PURPLE))
ax.text(5.0, 4.2, '신체활동량\n(매개변수 M)', ha='center', va='center', fontsize=13,
        fontweight='bold', bbox=dict(**box_kw, facecolor='#D5F5E3', edgecolor=C_TEAL))
ax.text(8.8, 2.5, 'BMI\n(종속변수)', ha='center', va='center', fontsize=13,
        fontweight='bold', bbox=dict(**box_kw, facecolor='#FDEBD0', edgecolor=C_CORAL))

# 경로 a
ax.annotate('', xy=(4.1, 3.9), xytext=(2.0, 2.9),
            arrowprops=dict(arrowstyle='->', color=C_TEAL, lw=2))
ax.text(2.8, 3.6, f'경로 a\nβ={results["a"]:.3f}', ha='center', fontsize=10,
        color=C_TEAL, fontweight='bold')

# 경로 b
ax.annotate('', xy=(7.9, 2.9), xytext=(5.9, 3.9),
            arrowprops=dict(arrowstyle='->', color=C_TEAL, lw=2))
ax.text(7.2, 3.6, f'경로 b\nβ={results["b"]:.3f}', ha='center', fontsize=10,
        color=C_TEAL, fontweight='bold')

# 경로 c (총효과)
ax.annotate('', xy=(7.9, 2.2), xytext=(2.1, 2.2),
            arrowprops=dict(arrowstyle='->', color=C_CORAL, lw=1.5, linestyle='dashed'))
ax.text(5.0, 1.8, f"직접효과 c'= {results["c'"]:.3f}  (총효과 c= {results['c']:.3f})",
        ha='center', fontsize=10, color=C_CORAL, fontweight='bold')

# 간접효과 라벨
ax.text(5.0, 0.8, f'간접효과 (a×b) = {indirect:.4f}',
        ha='center', fontsize=11, color=C_DARK, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#F8F9FA', edgecolor='#ADB5BD'))

ax.set_title('매개효과 경로 다이어그램\n우울감 → 신체활동 → BMI',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('plot1_mediation_diagram.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 1 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 2: 우울 심각도별 신체활동 & BMI 분포
# ════════════════════════════════════════════════════════
ORDER = ['없음(0-4)','경미(5-9)','중등도(10-14)','심각(15+)']
ORDER = [o for o in ORDER if o in df_an['DEP_GROUP'].values]
pal = [C_TEAL, '#55EFC4', C_CORAL, '#D63031']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('우울 심각도별 신체활동량 & BMI 분포', fontsize=14, fontweight='bold')

for ax, ycol, ylabel, colors in zip(
    axes, ['PA','BMI'],
    ['신체활동량 (분/주 또는 일수)','BMI (kg/m²)'],
    [pal, pal]
):
    data_bp = [df_an[df_an['DEP_GROUP']==g][ycol].values for g in ORDER]
    parts = ax.violinplot(data_bp, positions=range(len(ORDER)),
                          showmedians=True, showextrema=False)
    for pc, c in zip(parts['bodies'], colors):
        pc.set_facecolor(c); pc.set_alpha(0.7); pc.set_edgecolor('white')
    parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(2.5)

    # ANOVA
    f, p = stats.f_oneway(*data_bp)
    sig = f'F={f:.2f}, p{"<0.001" if p<0.001 else f"={p:.3f}"}'
    ax.set_title(f'{ylabel}\n({sig})', fontsize=11)
    ax.set_xticks(range(len(ORDER)))
    ax.set_xticklabels(ORDER, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xlabel('우울 심각도 그룹', fontsize=10)

plt.tight_layout()
plt.savefig('plot2_depression_group_violin.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 2 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 3: 산점도 3종 (X-M, M-Y, X-Y)
# ════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('변수 간 관계 산점도 + 회귀선', fontsize=14, fontweight='bold')

pairs = [
    ('PHQ9', 'PA',  'PHQ-9 (우울감)', '신체활동량', '경로 a: 우울 → 신체활동', C_TEAL),
    ('PA',   'BMI', '신체활동량', 'BMI', '경로 b: 신체활동 → BMI', C_TEAL),
    ('PHQ9', 'BMI', 'PHQ-9 (우울감)', 'BMI', '총효과 c: 우울 → BMI', C_CORAL),
]

for ax, (xc, yc, xl, yl, title, color) in zip(axes, pairs):
    sample = df_an.sample(min(1500, len(df_an)), random_state=42)
    ax.scatter(sample[xc], sample[yc], alpha=0.25, s=10,
               color=color, edgecolors='none')

    sl, ic, r, p, _ = stats.linregress(df_an[xc], df_an[yc])
    xr = np.linspace(df_an[xc].min(), df_an[xc].max(), 200)
    ax.plot(xr, sl*xr+ic, color=C_DARK, lw=2.2)

    p_str = '<0.001' if p < 0.001 else f'={p:.3f}'
    ax.set_title(f'{title}\nr={r:.3f}, p{p_str}', fontsize=10, fontweight='bold')
    ax.set_xlabel(xl, fontsize=10)
    ax.set_ylabel(yl, fontsize=10)

plt.tight_layout()
plt.savefig('plot3_scatter_paths.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 3 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 4: Bootstrap 간접효과 신뢰구간 시각화
# ════════════════════════════════════════════════════════
np.random.seed(42)
n_boot = 2000
boot_indirect = []

for _ in range(n_boot):
    samp = df_an.sample(len(df_an), replace=True)
    try:
        ma = smf.ols(f'PA ~ PHQ9{ctrl_str}', data=samp).fit()
        mb = smf.ols(f'BMI ~ PHQ9 + PA{ctrl_str}', data=samp).fit()
        boot_indirect.append(ma.params['PHQ9'] * mb.params['PA'])
    except:
        pass

boot_arr = np.array(boot_indirect)
ci_lo_b, ci_hi_b = np.percentile(boot_arr, [2.5, 97.5])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(boot_arr, bins=60, color=C_TEAL, edgecolor='white',
        linewidth=0.5, alpha=0.85)
ax.axvline(indirect, color=C_DARK, lw=2.5, label=f'간접효과 = {indirect:.4f}')
ax.axvline(ci_lo_b, color=C_CORAL, lw=2, ls='--',
           label=f'95% CI 하한 = {ci_lo_b:.4f}')
ax.axvline(ci_hi_b, color=C_CORAL, lw=2, ls='--',
           label=f'95% CI 상한 = {ci_hi_b:.4f}')
ax.axvline(0, color='gray', lw=1.2, ls=':', alpha=0.7, label='0 기준선')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1000],
                  ci_lo_b, ci_hi_b, alpha=0.15, color=C_CORAL)

sig_label = 'CI에 0 미포함 → 매개효과 유의' \
    if (ci_lo_b > 0 or ci_hi_b < 0) else 'CI에 0 포함 → 매개효과 비유의'
ax.set_title(f'Bootstrap 간접효과 분포 (n={n_boot})\n{sig_label}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('간접효과 (a × b)', fontsize=11)
ax.set_ylabel('빈도', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot4_bootstrap_ci.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✅ 그래프 4 저장')
print(f'   간접효과 = {indirect:.4f}  95%CI [{ci_lo_b:.4f}, {ci_hi_b:.4f}]  → {sig_label}')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 5: 효과 크기 분해 막대 (총효과 = 직접 + 간접)
# ════════════════════════════════════════════════════════
direct_e  = results["c'"]
total_e   = results['c']
indirect_e = indirect
pct_mediated = (indirect_e / total_e * 100) if total_e != 0 else 0

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('효과 크기 분해 & 매개 비율', fontsize=14, fontweight='bold')

# 왼쪽: 효과 분해 막대
ax = axes[0]
effects = {'총효과(c)': total_e, '직접효과(c\')': direct_e, '간접효과(a×b)': indirect_e}
colors_e = [C_DARK, C_CORAL, C_TEAL]
bars = ax.barh(list(effects.keys()), list(effects.values()),
               color=colors_e, edgecolor='white', height=0.5)
for bar, val in zip(bars, effects.values()):
    ax.text(val + 0.001 if val >= 0 else val - 0.001,
            bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=11, fontweight='bold')
ax.axvline(0, color='gray', lw=1)
ax.set_xlabel('회귀계수 β', fontsize=11)
ax.set_title('총효과 = 직접효과 + 간접효과', fontsize=11)

# 오른쪽: 매개 비율 파이
ax2 = axes[1]
if total_e != 0:
    pct_d = abs(direct_e / total_e) * 100
    pct_i = abs(indirect_e / total_e) * 100
    wedges, texts, autotexts = ax2.pie(
        [pct_i, pct_d],
        labels=['간접효과\n(신체활동 매개)', '직접효과\n(기타 경로)'],
        colors=[C_TEAL, C_CORAL],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    for at in autotexts: at.set_fontsize(12); at.set_fontweight('bold')
    ax2.set_title(f'총효과 중 신체활동 매개 비율\n(매개 비율 = {pct_mediated:.1f}%)', fontsize=11)

plt.tight_layout()
plt.savefig('plot5_effect_decompose.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 5 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 6: 상관관계 히트맵
# ════════════════════════════════════════════════════════
num_cols = [c for c in ['PHQ9','PA','BMI','AGE','INCOME'] if c in df_an.columns]
label_map = {'PHQ9':'우울감\n(PHQ-9)', 'PA':'신체활동량',
             'BMI':'BMI', 'AGE':'연령', 'INCOME':'소득수준'}

corr = df_an[num_cols].corr()
corr.index   = [label_map.get(c,c) for c in corr.index]
corr.columns = [label_map.get(c,c) for c in corr.columns]

cmap = LinearSegmentedColormap.from_list('custom',
       [C_CORAL, '#FFFFFF', C_TEAL], N=256)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt='.3f', cmap=cmap,
            center=0, vmin=-1, vmax=1, mask=mask,
            linewidths=1, linecolor='white', square=True,
            annot_kws={'size':12,'weight':'bold'},
            cbar_kws={'shrink':0.8}, ax=ax)
ax.set_title('변수 간 상관관계 히트맵', fontsize=13, fontweight='bold', pad=14)
ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig('plot6_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 6 저장')

## STEP 6. 최종 결과 요약 & 저장

In [ ]:
print('=' * 65)
print('📋 최종 연구 결과 요약')
print('=' * 65)
print(f'분석 대상: {len(df_an):,}명 (NHANES 2013-2014, 성인 20세+)')
print()
print('[1단계] 우울감 → BMI (총효과)')
print(f'  β = {results["c"]:.4f}, p = {results["c_p"]:.4f}')
print()
print('[2단계] 우울감 → 신체활동 (경로 a)')
print(f'  β = {results["a"]:.4f}, p = {results["a_p"]:.4f}')
print()
print('[3단계] 신체활동 → BMI (경로 b, X 통제 후)')
print(f'  β = {results["b"]:.4f}, p = {results["b_p"]:.4f}')
print()
print('[4단계] 간접효과 (a × b)')
print(f'  간접효과 = {indirect:.4f}')
print(f'  Bootstrap 95%CI = [{ci_lo_b:.4f}, {ci_hi_b:.4f}]')
print(f'  → {sig_label}')
print()
print(f'직접효과 (c\') = {direct_e:.4f}')
print(f'매개 비율     = {pct_mediated:.1f}%')
print(f'매개 유형     = {mediation_type}')
print()
print('정책적 함의:')
print('  우울감이 신체활동 감소를 통해 BMI에 영향을 미치므로')
print('  비만 예방을 위해 정신건강 개입이 선행되어야 함')
print('=' * 65)

# 저장
df_an.to_csv('분석데이터_최종.csv', index=False, encoding='utf-8-sig')
print('\n📁 저장 완료:')
saved = ['분석데이터_최종.csv'] + \
        [f'plot{i}_{n}.png' for i,n in enumerate(
         ['mediation_diagram','depression_group_violin',
          'scatter_paths','bootstrap_ci',
          'effect_decompose','heatmap'], 1)]
for f in saved:
    if os.path.exists(f):
        print(f'  ✔ {f}  ({os.path.getsize(f)//1024} KB)')

print('\n✅ 모든 분석 완료!')

---
## 📋 그래프 목록
| 파일 | 내용 |
|------|------|
| plot1_mediation_diagram | 매개효과 경로 다이어그램 (a, b, c' 계수 표시) |
| plot2_depression_group_violin | 우울 심각도별 신체활동/BMI 바이올린 플롯 |
| plot3_scatter_paths | 경로 a, b, c 산점도 + 회귀선 3종 |
| plot4_bootstrap_ci | Bootstrap 간접효과 분포 + 95% CI |
| plot5_effect_decompose | 총효과 분해 막대 + 매개비율 파이차트 |
| plot6_heatmap | 변수 간 상관관계 히트맵 |

## 논문 작성 시 참고
- **제목**: 성인의 우울감이 신체활동을 매개로 BMI에 미치는 영향
- **데이터**: NHANES 2013-2014 (N = 분석 결과 참고)
- **방법**: Baron & Kenny(1986) 4단계 + Bootstrap 간접효과 검증
- **한계**: 횡단연구로 인과관계 직접 확인 불가 / 미국 데이터